# 🎲 Multiple Testing
**ISLP Chapter 13 · Pattern Recognition for the Rest of Us**

> Test 100 hypotheses at α=0.05 and you expect 5 false positives just by chance — even if nothing is real. Multiple testing procedures control this inflation of false discoveries.

### What you'll learn
- The multiple testing problem: why p-values get inflated
- Family-wise error rate (FWER) and Bonferroni correction
- False discovery rate (FDR) and the Benjamini-Hochberg procedure
- When to use FWER vs FDR
- Practical applications: genomics, A/B testing, feature selection

### Time: ~45 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('✓ Setup complete')
from statsmodels.stats.multitest import multipletests
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
print('✓ Setup complete')

## 🎯 Part 1 — The Problem: False Positives at Scale

At α=0.05, each test has a 5% chance of a false positive. If we run m independent tests, the probability of *at least one* false positive is:

```
P(at least one false positive) = 1 - (1-0.05)^m
```

For m=100 tests: 1 - 0.95^100 ≈ 99.4% chance of at least one false positive!

In [ ]:
# Simulate: false positive inflation
m_values = [1, 5, 10, 20, 50, 100, 200, 500, 1000]
alpha = 0.05
fwer = [1 - (1-alpha)**m for m in m_values]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(m_values, fwer, 'o-', color='#e85d20', lw=2.5, markersize=7)
ax.axhline(0.05, color='#888', lw=1, ls='--', label='Single test α=0.05')
ax.set_xlabel('Number of tests (m)')
ax.set_ylabel('Probability of at least one false positive')
ax.set_xscale('log')
ax.set_title('Multiple Testing Problem: FWER Inflation')
ax.legend()
plt.tight_layout(); plt.show()
print('📌 With 100 tests at α=0.05, there is a 99.4% chance of at least one false positive!')
print('   Bonferroni or BH correction is needed')

In [ ]:
# Concrete simulation
np.random.seed(42)
m = 100  # tests
n = 30   # observations per test

# All 100 null hypotheses are TRUE (no real effect)
# Any rejection is a false positive
p_values_null = [stats.ttest_ind(
    np.random.normal(0,1,n), np.random.normal(0,1,n)
).pvalue for _ in range(m)]

fig, ax = plt.subplots(figsize=(10, 3))
colors = ['#e85d20' if p < 0.05 else '#1e5fa8' for p in p_values_null]
ax.bar(range(m), sorted(p_values_null), color=[c for _,c in sorted(zip(p_values_null, colors))],
      edgecolor='white', width=0.9)
ax.axhline(0.05, color='black', lw=1.5, ls='--', label='α=0.05')
fp = sum(p < 0.05 for p in p_values_null)
ax.set_title(f'100 tests, ALL null true — {fp} false positives (orange) at α=0.05')
ax.set_xlabel('Test (sorted by p-value)'); ax.set_ylabel('p-value')
ax.legend()
plt.tight_layout(); plt.show()
print(f'False positives: {fp} out of {m} tests (expected: {m*0.05:.0f})')

In [ ]:
# Bonferroni vs BH correction
# Now: 80 true nulls, 20 real effects
np.random.seed(1)
m_null = 80; m_alt = 20; n = 30
p_null = [stats.ttest_ind(np.random.normal(0,1,n), np.random.normal(0,1,n)).pvalue for _ in range(m_null)]
p_alt  = [stats.ttest_ind(np.random.normal(0,1,n), np.random.normal(2,1,n)).pvalue for _ in range(m_alt)]
p_all  = np.array(p_null + p_alt)
true_null = np.array([True]*m_null + [False]*m_alt)

results_corr = {}
for method, label in [('bonferroni','Bonferroni (FWER)'), ('fdr_bh','Benjamini-Hochberg (FDR)')]:
    reject, p_adj, _, _ = multipletests(p_all, alpha=0.05, method=method)
    tp = np.sum(reject & ~true_null)  # true positives
    fp = np.sum(reject & true_null)   # false positives
    fn = np.sum(~reject & ~true_null) # false negatives
    results_corr[label] = {'Rejected': reject.sum(), 'True Pos': tp, 'False Pos': fp, 'False Neg': fn, 'Power': tp/m_alt}
    print(f'{label}:')
    print(f'  Rejected: {reject.sum()} | TP={tp} | FP={fp} | FN={fn} | Power={tp/m_alt:.2f}')

print('\n📌 Bonferroni is very conservative — avoids false positives but misses many real effects')
print('   BH-FDR is more powerful — accepts some false positives to find more true effects')

In [ ]:
# Visualize: BH procedure step by step
sorted_p = np.sort(p_all)
m_total = len(p_all)
bh_threshold = np.arange(1, m_total+1) * 0.05 / m_total

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(range(1, m_total+1), sorted_p, s=20, alpha=0.6, label='p-values (sorted)',
          c=['#1a7a45' if p < 0.05 else '#888' for p in sorted(p_all)])
ax.plot(range(1, m_total+1), bh_threshold, color='#e85d20', lw=2, label='BH threshold k×α/m')
bonf_threshold = 0.05/m_total
ax.axhline(bonf_threshold, color='#c0392b', lw=1.5, ls='--', label=f'Bonferroni α/m={bonf_threshold:.4f}')
bh_reject_idx = np.where(sorted_p <= bh_threshold)[0]
if len(bh_reject_idx): ax.axvline(bh_reject_idx[-1]+1, color='#e85d20', lw=1, ls=':')
ax.set_xlabel('Rank (k)'); ax.set_ylabel('p-value')
ax.set_title('Benjamini-Hochberg Procedure: Reject all p ≤ k×α/m up to largest k')
ax.legend(); ax.set_ylim(0, 0.2)
plt.tight_layout(); plt.show()

In [ ]:
# Exercise workspace
# Task 1: Load a real genomics-style dataset and apply multiple testing correction
# Simulate 1000 gene expression comparisons with 50 truly differentially expressed genes
np.random.seed(42)
m_genes = 1000; m_de = 50
p_genes_null = [stats.ttest_ind(np.random.normal(0,1,20),np.random.normal(0,1,20)).pvalue for _ in range(m_genes-m_de)]
p_genes_de   = [stats.ttest_ind(np.random.normal(0,1,20),np.random.normal(1.5,1,20)).pvalue for _ in range(m_de)]
p_genes_all  = np.array(p_genes_null + p_genes_de)
# YOUR CODE HERE: apply Bonferroni and BH. Compare TP, FP, power.

# Task 2: A/B testing scenario — you tested 20 website variants simultaneously
# p-values from each test are given below. Which pass Bonferroni? Which pass BH at q=0.05?
ab_pvalues = [0.001, 0.023, 0.041, 0.087, 0.12, 0.18, 0.002, 0.045, 0.22, 0.008,
              0.19, 0.31, 0.005, 0.11, 0.67, 0.043, 0.28, 0.0003, 0.55, 0.032]
# YOUR CODE HERE

In [ ]:
# @title 📝 Quiz — Multiple Testing
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is FWER and what does Bonferroni control?
# @markdown **Q2:** What is FDR and what does Benjamini-Hochberg control?
# @markdown **Q3:** When would you choose FDR correction over FWER correction?
# @markdown **Q4:** With 1000 tests at α=0.05, how many false positives do you expect without correction?
# @markdown **Q5:** What is the Bonferroni correction threshold for 50 tests at α=0.05?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Multiple Testing"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


## 📚 Further Reading
- [ISLP Ch 13](https://intro-stat-learning.github.io/ISLP/) — Multiple Testing
- [statsmodels multipletests](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html)
- [Cross-Validation notebook](cross-validation.ipynb) — another way to control model selection error

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*